In [0]:
# 1. Setup your paths and names
CATALOG = "workspace" 
SCHEMA = "ais_db"
VOLUME = "ais_raw_data"

# The source is now the Bronze TABLE
target_bronze_table = f"{CATALOG}.{SCHEMA}.bronze_ais_table"

# NEW Checkpoint for the Silver stream (different from the Bronze one!)
silver_checkpoint_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoints/ais_silver/"

# The final destination
target_silver_table = f"{CATALOG}.{SCHEMA}.silver_ais_table"

# 2. Start the STREAM from the Bronze Table
# Spark looks at the Bronze Delta Log to find new data
bronze_stream_df = spark.readStream.table(target_bronze_table)

# 3. Apply your cleaning logic
from pyspark.sql.functions import col, regexp_replace, to_timestamp

silver_stream_df = (
    bronze_stream_df
    .select(
        col("MetaData.MMSI").cast("long").alias("mmsi"),
        col("MetaData.ShipName").alias("ship_name"),
        # Using your corrected timestamp logic
        to_timestamp(
            regexp_replace(col("MetaData.time_utc"), " \\+0000 UTC$", ""),
            "yyyy-MM-dd HH:mm:ss.SSSSSSSSS"
        ).alias("timestamp_utc"),
        col("Message.PositionReport.Latitude").cast("double").alias("latitude"),
        col("Message.PositionReport.Longitude").cast("double").alias("longitude"),
        col("Message.PositionReport.Sog").cast("double").alias("speed_knots")
    )
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull() & col("ship_name").isNotNull())
)

# 4. Write the STREAM to the Silver Table
query = (silver_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", silver_checkpoint_path)
    .option("mergeSchema", "true")  # Allows the Silver table to grow if columns change
    .trigger(availableNow=True)    # Process all new data and then stop the cluster
    .toTable(target_silver_table))

query.awaitTermination()

print(f"✅ Silver table {target_silver_table} updated!")

In [0]:
display(spark.read.table("workspace.ais_db.silver_ais_table"))